# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Selected Lane: Content Refresh Queue

ML Task Type: Learning to Rank (LTR) / Priority Scoring

Why this type?

The goal is not simply to classify whether a page is declining (0 or 1), but to order all client pages by priority for editorial refresh. Editorial teams have limited bandwidth (e.g., updating 50 pages per month), so the model must rank pages based on the urgency and expected impact of a refresh.


In [10]:
# Task framing: Ranking / Priority Scoring

## 2. Target or proxy

What we predict (Target / Proxy):Since historical records of manual content refreshes are unlabelled, we construct a defined outcome proxy based on 90-day forward organic search decay:Proxy Label ($y$): target_proxy_decay (1 if organic clicks/impressions dropped by >30% over a 90-day window, 0 otherwise).Source: Derived directly from observed performance metrics in Google Search Console data over time.


In [11]:
# Proxy definition: 90-day search traffic decay (>30% loss)

## 3. Success metric

Primary ML Metric: NDCG@50 (Normalized Discounted Cumulative Gain at K=50) and Precision@50.

Why this metric?

Since content teams can only refresh a subset of pages (e.g., top 50), NDCG@50 measures how accurately the model places the highest-decayed pages at the very top of the priority list.

Business Metric: Recovered organic traffic volume per editorial hour spent updating content.


In [12]:
# Primary Metric: NDCG@50 / Precision@50

## 4. The unit of analysis, as a real dataframe

Unit of Analysis Summary:

One Row = One URL / Page snapshot.

The target column target_proxy_decay grounds the learning problem: 1 indicates measurable traffic decay requiring attention, 0 indicates healthy or stable traffic.

In [13]:
import pandas as pd
import numpy as np
import os

if not os.path.exists("data"):
    !git clone https://github.com/HamzaMohamed25/flyrank-ml-internship.git temp_repo
    !cp -r temp_repo/data .
    !rm -rf temp_repo

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df['target_proxy_decay'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print(f"Dataset Shape: {df.shape}")
print(f"Unit of Analysis: Single Content Item (Unique Items: {df['content_id'].nunique():,})\n")

cols_to_display = ['content_id', 'days_since_last_update', 'trend_direction', 'target_proxy_decay']
display(df[cols_to_display].head(5))

Dataset Shape: (30000, 45)
Unit of Analysis: Single Content Item (Unique Items: 30,000)



,content_id,days_since_last_update,trend_direction,target_proxy_decay
0,content_304f48230142,20,down,1
1,content_a1fb4e703a9e,25,down,1
2,content_9aa793d4d895,20,down,1
3,content_331d6c4de07b,22,stable,0
4,content_d99b7a2d90ca,14,down,1


## 5. Why ML beats a fixed rule here

Why simple if-statements fail:

Complex feature interactions: A fixed rule like "refresh if age > 180 days" fails because a 300-day-old page might still rank #1 with high organic clicks, while a 60-day-old page might have suffered an algorithm hit or intent drift.

Multi-signal weighting: ML dynamically weighs multiple subtle features simultaneously (CTR velocity, impression trend drops, position volatility, seasonality, and page age) to output a calibrated priority score rather than a blunt rule.

In [14]:
# ML vs Heuristics: Captures non-linear interactions across age, CTR, and impressions

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.